# RS Matrix Factorization Tavsiye(Oneri) Sitemleri: Film Tavsiyesi

In [1]:
import pandas as pd

In [2]:
ratings=pd.read_csv('spark/ml-100k/u.data', sep='\t', header=None, names=['user_id','movie_id','rating'],usecols=range(3))

In [3]:
ratings

,user_id,movie_id,rating
0,0,50,5
1,0,172,5
2,0,133,1
3,196,242,3
4,186,302,3
...,...,...,...
99998,880,476,3
99999,716,204,5
100000,276,1090,1
100001,13,225,2


In [4]:
movies=pd.read_csv('spark/ml-100k/u.item', encoding='iso-8859-1', sep='|', header=None, names=['movie_id','title'],usecols=range(2))

In [5]:
movies

,movie_id,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)
...,...,...
1677,1678,Mat' i syn (1997)
1678,1679,B. Monkey (1998)
1679,1680,Sliding Doors (1998)
1680,1681,You So Crazy (1994)


In [6]:
user=pd.read_csv('spark/ml-100k/u.user', sep='|', header=None, names=['user_id','age','gender','profession','zipcode'])

In [7]:
user

,user_id,age,gender,profession,zipcode
0,1,24,M,technician,85711
1,2,53,F,other,94043
2,3,23,M,writer,32067
3,4,24,M,technician,43537
4,5,33,F,other,15213
...,...,...,...,...,...
938,939,26,F,student,33319
939,940,32,M,administrator,02215
940,941,20,M,student,97229
941,942,48,F,librarian,78209


In [8]:
rating=pd.merge(movies,ratings)

In [9]:
rating

,movie_id,title,user_id,rating
0,1,Toy Story (1995),308,4
1,1,Toy Story (1995),287,5
2,1,Toy Story (1995),148,4
3,1,Toy Story (1995),280,4
4,1,Toy Story (1995),66,3
...,...,...,...,...
99998,1678,Mat' i syn (1997),863,1
99999,1679,B. Monkey (1998),863,3
100000,1680,Sliding Doors (1998),863,2
100001,1681,You So Crazy (1994),896,3


In [10]:
rating.title.value_counts()

title
Star Wars (1977)                        584
Contact (1997)                          509
Fargo (1996)                            508
Return of the Jedi (1983)               507
Liar Liar (1997)                        485
                                       ... 
Invitation, The (Zaproszenie) (1986)      1
Symphonie pastorale, La (1946)            1
Nothing Personal (1995)                   1
Ripe (1996)                               1
Brother's Kiss, A (1997)                  1
Name: count, Length: 1664, dtype: int64

### Buradan sonrasini RSMatrixFactorization'a gore yapacagiz

In [12]:
mf=rating.pivot_table(index=['user_id'],columns=['title'],values='rating')

In [16]:
mf.tail()

title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
user_id,,,,,,,,,,,,,,,,,,,,,
939,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
940,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
941,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
942,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
943,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,4.0,3.0,NaN,NaN,NaN,NaN


In [18]:
swr=mf['Star Wars (1977)']

In [19]:
swr.value_counts()

Star Wars (1977)
5.0    326
4.0    176
3.0     57
2.0     16
1.0      9
Name: count, dtype: int64

In [21]:
mf[['Star Wars (1977)','101 Dalmatians (1996)']].corr()

title,Star Wars (1977),101 Dalmatians (1996)
title,,
Star Wars (1977),1.000000,0.211132
101 Dalmatians (1996),0.211132,1.000000


In [23]:
sm=mf.corrwith(swr)

In [24]:
sm.sort_values(ascending=False)

title
Hollow Reed (1996)                         1.0
Stripes (1981)                             1.0
No Escape (1994)                           1.0
Man of the Year (1995)                     1.0
Cosi (1996)                                1.0
                                          ... 
Wonderland (1997)                          NaN
Wooden Man's Bride, The (Wu Kui) (1994)    NaN
Yankee Zulu (1994)                         NaN
You So Crazy (1994)                        NaN
Á köldum klaka (Cold Fever) (1994)         NaN
Length: 1664, dtype: float64

In [25]:
import numpy as np

In [30]:
mstats=rating.groupby('title').agg({'rating':[np.size , np.mean]})

C:\Users\kosey\AppData\Local\Temp\ipykernel_2728\257918851.py:1: FutureWarning: The provided callable <function mean at 0x000002CAC0D5F4C0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  mstats=rating.groupby('title').agg({'rating':[np.size , np.mean]})


In [31]:
mstats

rating          
                                        size      mean
title                                                 
'Til There Was You (1997)                  9  2.333333
1-900 (1994)                               5  2.600000
101 Dalmatians (1996)                    109  2.908257
12 Angry Men (1957)                      125  4.344000
187 (1997)                                41  3.024390
...                                      ...       ...
Young Guns II (1990)                      44  2.772727
Young Poisoner's Handbook, The (1995)     41  3.341463
Zeus and Roxanne (1997)                    6  2.166667
unknown                                    9  3.444444
Á köldum klaka (Cold Fever) (1994)         1  3.000000

[1664 rows x 2 columns]

In [32]:
pm=mstats[mstats['rating']['size']>100]

In [36]:
smdata=pd.DataFrame(sm,columns=['similarity'])

In [37]:
pm.columns=pm.columns.get_level_values(0)
df=pm.join(smdata)

In [38]:
df.sort_values(['similarity'],ascending=False)

,rating,rating,similarity
title,,,
Star Wars (1977),584,4.359589,1.000000
"Empire Strikes Back, The (1980)",368,4.206522,0.748353
Return of the Jedi (1983),507,4.007890,0.672556
Raiders of the Lost Ark (1981),420,4.252381,0.536117
Austin Powers: International Man of Mystery (1997),130,3.246154,0.377433
...,...,...,...
"Edge, The (1997)",113,3.539823,-0.127167
As Good As It Gets (1997),112,4.196429,-0.130466
Crash (1996),128,2.546875,-0.148507
